# Task 5 — Data Quality and Reconciliation

Dedicated quality gate that runs after the Gold layer is built. Executes every critical rule (Bronze ingestion, Silver primary keys, fact surrogate keys, numeric business rules, Gold-vs-Silver reconciliation within 0.01% tolerance, rejected-record reporting) and **raises a pipeline-stopping error if any critical check fails**. Full results are persisted to the ops schema in `dq_pipeline_report`.

In [0]:
# -----------------------------------------------------------------------------
# Quality-gate framework: every check is recorded (even failures) and the
# notebook raises a pipeline-stopping error at the end if any CRITICAL check failed.
# -----------------------------------------------------------------------------
from pyspark.sql import functions as F, types as T

try:
    catalog = dbutils.widgets.get("catalog_name")
except Exception:
    dbutils.widgets.text("catalog_name", "dev")
    catalog = dbutils.widgets.get("catalog_name")

BRONZE = f"{catalog}.bronze"
SILVER = f"{catalog}.silver"
GOLD = f"{catalog}.gold"
OPS = f"{catalog}.ops"

TABLES = ["region", "nation", "airline", "route", "airline_route",
          "passengers", "booking", "flight_booking"]

dq_results = []   # (category, check_name, severity, detail, status)


def record(category, check_name, severity, detail, passed):
    status = "PASS" if passed else "FAIL"
    dq_results.append((category, check_name, severity, detail, status))
    print(f"[{status}] ({severity:8s}) {category} :: {check_name} :: {detail}")


def null_rows(df, cols):
    """Count rows where ANY of the given columns is null."""
    cond = None
    for c in cols:
        bad = F.col(c).isNull()
        cond = bad if cond is None else (cond | bad)
    return df.filter(cond).count()

In [0]:
# -----------------------------------------------------------------------------
# CHECK 1 - All eight source tables were ingested into Bronze (CRITICAL)
# -----------------------------------------------------------------------------
for t in TABLES:
    try:
        n = spark.table(f"{BRONZE}.{t}").count()
        record("1_bronze_ingestion", f"bronze.{t} ingested", "CRITICAL",
               f"{n} rows present", n > 0)
    except Exception:
        record("1_bronze_ingestion", f"bronze.{t} ingested", "CRITICAL",
               "table missing or unreadable", False)

# -----------------------------------------------------------------------------
# CHECK 2 - Primary keys are populated and unique where required (CRITICAL)
# -----------------------------------------------------------------------------
pk_spec = [
    ("region", ["r_regionkey"]),
    ("nation", ["n_nationkey"]),
    ("airline", ["al_airlineid"]),
    ("route", ["ro_routeid"]),
    ("airline_route", ["ar_airlineid", "ar_routeid"]),
    ("passengers", ["pa_passengerid"]),
    ("booking", ["bk_bookingid"]),
    ("flight_booking", ["fb_bookingid", "fb_linenumber"]),
]
for t, pks in pk_spec:
    df = spark.table(f"{SILVER}.{t}")
    nulls = null_rows(df, pks)
    dups = df.groupBy(*pks).count().filter(F.col("count") > 1).count()
    record("2_primary_keys", f"silver.{t} PK populated", "CRITICAL",
           f"PK ({', '.join(pks)}): {nulls} null rows", nulls == 0)
    record("2_primary_keys", f"silver.{t} PK unique", "CRITICAL",
           f"PK ({', '.join(pks)}): {dups} duplicate key groups", dups == 0)

In [0]:
# -----------------------------------------------------------------------------
# CHECK 3 - Fact table contains no null surrogate/dimension keys (CRITICAL)
# -----------------------------------------------------------------------------
fact = spark.table(f"{GOLD}.fact_bookings")

key_cols = ["booking_id", "line_number", "passenger_key", "route_key",
            "airline_key", "home_geography_key", "airline_geography_key"]
for k in key_cols:
    nulls = fact.filter(F.col(k).isNull()).count()
    record("3_fact_keys", f"fact_bookings.{k} not null", "CRITICAL",
           f"{nulls} null values", nulls == 0)

# -----------------------------------------------------------------------------
# CHECK 4 - Numeric measures satisfy the documented business rules (CRITICAL):
#           ticket price > 0, discount in [0, 1], seats >= 1
# -----------------------------------------------------------------------------
sil_fb = spark.table(f"{SILVER}.flight_booking")

numeric_rules = [
    ("ticket_price > 0",
     F.col("fb_ticketprice").isNull() | (F.col("fb_ticketprice") <= 0)),
    ("discount in [0, 1]",
     F.col("fb_discount").isNull() | (F.col("fb_discount") < 0) | (F.col("fb_discount") > 1)),
    ("seats >= 1",
     F.col("fb_seats").isNull() | (F.col("fb_seats") < 1)),
]
for rule_name, viol in numeric_rules:
    bad = sil_fb.filter(viol).count()
    record("4_numeric_rules", f"silver.flight_booking: {rule_name}", "CRITICAL",
           f"{bad} violating rows", bad == 0)

for rule_name, viol in [
    ("ticket_price > 0", F.col("ticket_price").isNull() | (F.col("ticket_price") <= 0)),
    ("discount in [0, 1]", F.col("discount_rate").isNull() | (F.col("discount_rate") < 0) | (F.col("discount_rate") > 1)),
    ("seats >= 1", F.col("seats").isNull() | (F.col("seats") < 1)),
]:
    bad = fact.filter(viol).count()
    record("4_numeric_rules", f"gold.fact_bookings: {rule_name}", "CRITICAL",
           f"{bad} violating rows", bad == 0)

In [0]:
# -----------------------------------------------------------------------------
# CHECK 5 - Gold revenue and row counts reconcile with Silver, tolerance <= 0.01% (CRITICAL)
# -----------------------------------------------------------------------------
sil_fb = spark.table(f"{SILVER}.flight_booking")

gold_rows = fact.count()
silver_rows = sil_fb.count()
row_diff_pct = abs(gold_rows - silver_rows) / silver_rows * 100 if silver_rows else (0.0 if gold_rows == 0 else float("inf"))
record("5_gold_silver_reconciliation", "row count within 0.01%", "CRITICAL",
       f"gold={gold_rows} silver={silver_rows} diff={row_diff_pct:.6f}%", row_diff_pct <= 0.01)

# Official metric: net revenue (before tax) = fb_ticketprice * (1 - fb_discount)
gold_rev = float(fact.agg(F.sum("net_revenue")).collect()[0][0] or 0)
silver_rev = float(sil_fb.agg(
    F.sum(F.col("fb_ticketprice") * (1 - F.col("fb_discount")))).collect()[0][0] or 0)
rev_diff_pct = abs(gold_rev - silver_rev) / abs(silver_rev) * 100 if silver_rev else (0.0 if gold_rev == 0 else float("inf"))
record("5_gold_silver_reconciliation", "net revenue within 0.01%", "CRITICAL",
       f"gold={gold_rev:.2f} silver={silver_rev:.2f} diff={rev_diff_pct:.6f}%", rev_diff_pct <= 0.01)

# -----------------------------------------------------------------------------
# CHECK 6 - Rejected-record counts per table are reported, and every Bronze row
#           is accounted for: bronze = silver + quarantine (CRITICAL balance)
# -----------------------------------------------------------------------------
rej_rows = []
for t in TABLES:
    b = spark.table(f"{BRONZE}.{t}").count()
    s = spark.table(f"{SILVER}.{t}").count()
    q = spark.table(f"{SILVER}.{t}_quarantine").count()
    balanced = (b == s + q)
    record("6_rejected_records", f"{t} bronze = silver + quarantine", "CRITICAL",
           f"bronze={b} silver={s} quarantine={q}", balanced)
    rej_rows.append((t, b, s, q, b - s, b - s - q))

rej_schema = T.StructType([
    T.StructField("table_name", T.StringType()),
    T.StructField("bronze_rows", T.LongType()),
    T.StructField("silver_rows", T.LongType()),
    T.StructField("rejected_rows", T.LongType()),
    T.StructField("reject_pct_of_bronze", T.DoubleType()),
    T.StructField("unaccounted_rows", T.LongType()),
])
rejected_df = spark.createDataFrame(
    [(t, b, s, q, round((b - s) / b * 100, 4) if b else 0.0, b - s - q) for t, b, s, q, _, _ in rej_rows],
    rej_schema)
rejected_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OPS}.rejected_record_counts")
print("Rejected-record counts written to", f"{OPS}.rejected_record_counts")
display(rejected_df)

In [0]:
# -----------------------------------------------------------------------------
# FINAL GATE - persist the report and fail the pipeline if any CRITICAL check failed
# -----------------------------------------------------------------------------
report_schema = T.StructType([
    T.StructField("category", T.StringType()),
    T.StructField("check_name", T.StringType()),
    T.StructField("severity", T.StringType()),
    T.StructField("detail", T.StringType()),
    T.StructField("status", T.StringType()),
])
report_df = spark.createDataFrame(dq_results, report_schema)
report_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OPS}.dq_pipeline_report")
print(f"Data-quality report written to {OPS}.dq_pipeline_report ({report_df.count()} checks).")
display(report_df.orderBy("category", "check_name"))

failed = [(r[1], r[3]) for r in dq_results if r[4] == "FAIL" and r[2] == "CRITICAL"]
if failed:
    listing = "; ".join(f"{name} ({detail})" for name, detail in failed)
    raise Exception(f"PIPELINE-STOPPING DATA-QUALITY FAILURE: {len(failed)} critical check(s) failed -> {listing}")
print("ALL CRITICAL DATA-QUALITY CHECKS PASSED - the pipeline may proceed.")